# Stage A 训练 loss 对比（AE-3 vs BPv5 local4 vs dist8）

只读 `loss.log`，不加载模型、不用 GPU。绘图风格对齐 `stage_b_compare_b3_b5.ipynb`（浅色 raw + 1,000 步 trailing mean）。

- **L1** 全量步数叠图；**L2** 放大到与 BPv5 重叠的前 ~50k；**L3** local4 vs dist8 仅；末尾对齐步数表 + 原因分析。
- 导出到 `outputs/BPv5/stage_a/reports/loss_compare_ae3/`（同名覆盖）。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string(index=False) if isinstance(obj, pd.DataFrame) else obj)

REPO = Path('/vla/workspace/my_tbot')
STAGE_A = REPO / 'outputs/BPv5/stage_a'
COMPARE_DIR = STAGE_A / 'reports' / 'loss_compare_ae3'

RUNS = [
    {
        'name': 'AE-3',
        'loss_log': Path('/vla/workspace/code/files/ae3loss.log'),
        'note': 'bpv4cdx_ae · 0.8B · AE0.5 · act32 · causal_vlm=False',
    },
    {
        'name': 'BPv5-local4',
        'loss_log': STAGE_A / 'main' / '2026-11-13_14-46' / 'loss.log',
        'note': 'bpv5_qformer · Wall4B frozen · AE0.25 · act80 · causal_vlm=True · 4 GPU',
    },
    {
        'name': 'BPv5-dist8',
        'loss_log': STAGE_A / 'main_dist8' / '2026-11-13_22-56' / 'loss.log',
        'note': 'same recipe · 8 GPU distributed',
    },
]

SMOOTH_STEPS = 1000
WINDOW_STEPS = (100, 1000, 5000)
ZOOM_MAX = 50000
EXPORT = True
FORMATS = ('png', 'pdf', 'svg')
DPI = 300
FIGSIZE_ROW = (14, 4.3)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.22,
    'legend.fontsize': 8, 'figure.dpi': 110, 'figure.max_open_warning': 0,
    'savefig.facecolor': 'white', 'pdf.fonttype': 42, 'svg.fonttype': 'none',
})
pd.set_option('display.max_columns', 24)
pd.set_option('display.width', 200)

RUN_COLORS = {
    'AE-3': '#0072B2',
    'BPv5-local4': '#D55E00',
    'BPv5-dist8': '#009E73',
}
MARKERS = ('o', 's', '^')

In [ ]:
def load_loss(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    need = {'step', 'loss'}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f'{path}: missing columns {sorted(missing)}')
    df = df.sort_values('step').drop_duplicates('step', keep='last').reset_index(drop=True)
    df['step'] = df['step'].astype(int)
    df['loss'] = df['loss'].astype(float)
    return df


def infer_freq(df: pd.DataFrame) -> int:
    d = df['step'].diff().dropna()
    if d.empty:
        return 100
    return int(d.mode().iloc[0])


def train_smooth(df, metric, freq, width=SMOOTH_STEPS):
    k = width // freq
    series = df.set_index('step')[metric]
    steps = pd.Series(series.index, index=series.index)
    complete = steps - steps.shift(k - 1) == width - freq
    return series.rolling(k, min_periods=k).mean().where(complete).to_numpy()


CTX = {}
for spec in RUNS:
    path = Path(spec['loss_log'])
    if not path.is_file():
        print(f"{spec['name']}: 跳过——找不到 {path}")
        continue
    df = load_loss(path)
    freq = infer_freq(df)
    if SMOOTH_STEPS % freq:
        raise ValueError(f"{spec['name']}: SMOOTH_STEPS={SMOOTH_STEPS} 必须整除 log_freq={freq}")
    CTX[spec['name']] = {
        **spec,
        'df': df,
        'freq': freq,
        'path': path,
    }
    print(
        f"{spec['name']}: steps {int(df.step.min()):,}→{int(df.step.max()):,} "
        f"(n={len(df)}, freq={freq}); last loss={df.loss.iloc[-1]:.4f}; {spec['note']}"
    )

FIGURES = {}
NOTES = []


def register(name, fig, note=''):
    FIGURES[name] = fig
    NOTES.append({'figure': name, 'caption': note})
    plt.show()


def step_fmt(x, _pos=None):
    x = float(x)
    if abs(x) >= 1_000_000:
        return f'{x/1_000_000:.1f}M'
    if abs(x) >= 1000:
        return f'{x/1000:.0f}k'
    return f'{x:.0f}'

In [ ]:
# L1 — full horizon overlay
fig, ax = plt.subplots(figsize=FIGSIZE_ROW)
for i, (name, ctx) in enumerate(CTX.items()):
    df, freq, color = ctx['df'], ctx['freq'], RUN_COLORS[name]
    ax.plot(df['step'], df['loss'], color=color, lw=0.7, alpha=0.23,
            label=f'{name} logged {freq}-step mean (rank 0)')
    ax.plot(df['step'], train_smooth(df, 'loss', freq), color=color, lw=1.8,
            label=f'{name} trailing {SMOOTH_STEPS:,}-step mean')
ax.set_title('L1  Stage A train loss — AE-3 vs BPv5 local4 / dist8')
ax.set_xlabel('step')
ax.set_ylabel('flow MSE loss')
ax.xaxis.set_major_formatter(FuncFormatter(step_fmt))
ax.legend(loc='upper right', ncol=1, frameon=False)
fig.tight_layout()
register('L1_train_loss_full', fig,
         'Rank-0 loss.log; pale = logged point, solid = 1,000-step trailing mean.')

In [ ]:
# L2 — zoom to BPv5 horizon
fig, ax = plt.subplots(figsize=FIGSIZE_ROW)
for name, ctx in CTX.items():
    df = ctx['df']
    z = df[df['step'] <= ZOOM_MAX].copy()
    if z.empty:
        continue
    freq, color = ctx['freq'], RUN_COLORS[name]
    ax.plot(z['step'], z['loss'], color=color, lw=0.7, alpha=0.23,
            label=f'{name} logged')
    # smooth on full then clip so early window matches full-run definition
    sm = train_smooth(df, 'loss', freq)
    sm_df = pd.DataFrame({'step': df['step'], 'sm': sm})
    sm_z = sm_df[sm_df['step'] <= ZOOM_MAX]
    ax.plot(sm_z['step'], sm_z['sm'], color=color, lw=1.8,
            label=f'{name} trailing {SMOOTH_STEPS:,}-step mean')
ax.set_xlim(0, ZOOM_MAX)
ax.set_title(f'L2  Same curves, zoomed to first {ZOOM_MAX:,} steps')
ax.set_xlabel('step')
ax.set_ylabel('flow MSE loss')
ax.xaxis.set_major_formatter(FuncFormatter(step_fmt))
ax.legend(loc='upper right', ncol=1, frameon=False)
fig.tight_layout()
register('L2_train_loss_zoom50k', fig,
         f'Overlay clipped to step ≤ {ZOOM_MAX:,} for fair early-stage comparison.')

In [ ]:
# L3 — local4 vs dist8 only
fig, ax = plt.subplots(figsize=FIGSIZE_ROW)
for name in ('BPv5-local4', 'BPv5-dist8'):
    if name not in CTX:
        continue
    ctx = CTX[name]
    df, freq, color = ctx['df'], ctx['freq'], RUN_COLORS[name]
    ax.plot(df['step'], df['loss'], color=color, lw=0.7, alpha=0.23,
            label=f'{name} logged {freq}-step mean')
    ax.plot(df['step'], train_smooth(df, 'loss', freq), color=color, lw=1.8,
            label=f'{name} trailing {SMOOTH_STEPS:,}-step mean')
ax.set_title('L3  BPv5 local4 vs dist8 (same recipe)')
ax.set_xlabel('step')
ax.set_ylabel('flow MSE loss')
ax.xaxis.set_major_formatter(FuncFormatter(step_fmt))
ax.legend(loc='upper right', frameon=False)
fig.tight_layout()
register('L3_bpv5_local_vs_dist8', fig,
         'If these two track, high loss is not a multi-GPU / DDP artifact.')

In [ ]:
def nearest_loss(df, step):
    i = (df['step'] - step).abs().idxmin()
    r = df.loc[i]
    return int(r['step']), float(r['loss'])


def trailing_at(df, freq, step, width=SMOOTH_STEPS):
    sm = train_smooth(df, 'loss', freq, width=width)
    i = (df['step'] - step).abs().idxmin()
    v = sm[df.index.get_loc(i) if False else list(df.index).index(i)]
    # more reliable: align by position
    pos = int(np.argmin(np.abs(df['step'].to_numpy() - step)))
    return float(sm[pos]) if sm[pos] == sm[pos] else float('nan')


align_steps = [1000, 5000, 10000, 20000, 24000, 45000, 50000]
rows = []
for s in align_steps:
    row = {'target_step': s}
    for name, ctx in CTX.items():
        df = ctx['df']
        if df['step'].max() < s * 0.9:
            row[f'{name}_step'] = None
            row[f'{name}_loss'] = None
            row[f'{name}_trail1k'] = None
            continue
        st, lo = nearest_loss(df, s)
        row[f'{name}_step'] = st
        row[f'{name}_loss'] = lo
        row[f'{name}_trail1k'] = trailing_at(df, ctx['freq'], s)
    rows.append(row)

table = pd.DataFrame(rows)
display(table.round(4))

# summary windows at end of each run
sum_rows = []
for name, ctx in CTX.items():
    df, freq = ctx['df'], ctx['freq']
    last = int(df['step'].iloc[-1])
    entry = {'run': name, 'last_step': last, 'last_loss': float(df['loss'].iloc[-1])}
    for w in WINDOW_STEPS:
        k = w // freq
        entry[f'mean_last_{w}'] = float(df['loss'].tail(k).mean()) if len(df) >= k else float('nan')
    sum_rows.append(entry)
summary = pd.DataFrame(sum_rows)
display(summary.round(4))

## 原因分析（相对 AE-3）

### 图上先看到的事实
1. **BPv5 local4 ≈ dist8**：同配方下两条曲线几乎重合 → **不是分布式 / DDP / 八卡同步问题**。
2. **曲线在降，但平台远高于 AE-3**：约 20k 步时 AE-3 ≈ 0.07，BPv5 ≈ 0.23–0.26（约 **3–4×**）；local4 到 ~49k 仍停在 ~0.23，而 AE-3 同进度已到 ~0.06。
3. 因此更像 **任务/配方天花板不同**，而不是「完全没在学」或「某台机器坏了」。

### 最可能的原因（按优先级）
1. **动作空间与 loss 归约不同（最硬）**  
   - AE-3：`max_action_dim=32`，`masked_flow_loss` 分母是 `valid_steps × A`（**含 pad 维**）。pad 上 target≈0、模型很快学会预测 0 → **把均值拉低**。  
   - BPv5：`max_action_dim=80` + `action.dim_mask`，`masked_flow_loss_dims` **只平均有效维** → 数值不可直接和 AE-3 比绝对大小；有效维更多、多本体异构，平台自然更高。
2. **数据更难**  
   AE-3 近似单域 / 小动作维；BPv5 Stage A 是 **80D 多源混合**（多机器人、多相机、异构有效维）。同样的 flow MSE，在多源上更难压到 0.05 量级。
3. **骨干与可训练容量**  
   - AE-3：0.8B、**双向** VLM（`causal_vlm=False`）、AE width **0.5**。  
   - BPv5：Wall **4B 全冻**、**因果** VLM（`causal_vlm=True`）+ image→text 前缀、AE width **0.25**。  
   表达动作主要靠更窄的 action expert + QFormer/segment query，**拟合速度与下限都会变差**。
4. **优化器日程不同，但不是主因**  
   BPv5 peak lr 更高（1e-4 vs AE-3 组内 1e-4 / scheduler 5e-5）；两边都在降，**local≈dist8** 说明 lr/多卡也不是「居高不下」的根因。
5. **不太像的原因**  
   单纯「本机坏了 / 八卡坏了」、loss 日志写错、完全没反传——会被 L3 重合与持续下降否定。

### 建议的下一步（验证用）
- 用 **HOM matched MSE**（与训练 flow 同维 mask）看是否也相对 AE-3 / 随机基线在降；若 matched 在降而 train loss 平台高，更支持「量纲/任务难」而非 bug。
- 做 **单源 / 32D 子集** 或临时关掉 dim-mask 改用 AE-3 同款归约，看 loss 是否立刻落到可比区间（区分「归约假象」vs「真学不动」）。
- 试 **AE width 0.5** 或短暂解冻少量 VLM/LoRA，看平台是否下移（容量假说）。

In [ ]:
if EXPORT:
    COMPARE_DIR.mkdir(parents=True, exist_ok=True)
    meta = {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'smooth_steps': SMOOTH_STEPS,
        'zoom_max': ZOOM_MAX,
        'runs': {
            name: {
                'path': str(ctx['path']),
                'note': ctx['note'],
                'freq': ctx['freq'],
                'step_min': int(ctx['df']['step'].min()),
                'step_max': int(ctx['df']['step'].max()),
                'last_loss': float(ctx['df']['loss'].iloc[-1]),
            }
            for name, ctx in CTX.items()
        },
        'notes': NOTES,
    }
    (COMPARE_DIR / 'meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    table.to_csv(COMPARE_DIR / 'aligned_steps.csv', index=False)
    summary.to_csv(COMPARE_DIR / 'run_summary.csv', index=False)
    for name, fig in FIGURES.items():
        for ext in FORMATS:
            fig.savefig(COMPARE_DIR / f'{name}.{ext}', dpi=DPI if ext == 'png' else None,
                        bbox_inches='tight')
    print(f'exported {len(FIGURES)} figures × {len(FORMATS)} formats → {COMPARE_DIR}')
else:
    print('EXPORT=False; skipped')